In [18]:
# 1. Leitura dos dados
import pandas as pd

df = pd.read_csv("history_mock.csv")

# 2. Análise rápida
print(df.info())
print(df['target'].value_counts())

# 3. Pré-processamento
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Seleção de features (ajuste conforme necessário)
features = ['from_stage', 'to_stage', 'total_moves', 'days_since_creation', 'current_stage_duration', 'value', 'sector', 'prioridade']
X = df[features]
y = df['target']

# Separar colunas categóricas e numéricas
cat_cols = ['from_stage', 'to_stage', 'sector', 'prioridade']
num_cols = ['total_moves', 'days_since_creation', 'current_stage_duration', 'value']

# OneHotEncoder para categóricos
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat = ohe.fit_transform(X[cat_cols])

# Padronização dos numéricos
scaler = StandardScaler()
X_num = scaler.fit_transform(X[num_cols])

# Concatenar features finais
import numpy as np
X_final = np.hstack([X_cat, X_num])

# Codificação do target
from sklearn.preprocessing import LabelEncoder
y_le = LabelEncoder()
y_enc = y_le.fit_transform(y)

# Split treino/teste
X_train, X_test, y_train, y_test = train_test_split(X_final, y_enc, stratify=y_enc, test_size=0.2, random_state=42)

# 4. Modelagem (Random Forest)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

model = RandomForestClassifier(n_estimators=20, max_depth=6, random_state=42)
model.fit(X_train, y_train)

# 5. Avaliação - Treino
y_train_pred = model.predict(X_train)
print("Métricas no Treino:")
print(classification_report(y_train, y_train_pred, target_names=y_le.classes_))
print(confusion_matrix(y_train, y_train_pred))

# 6. Avaliação - Teste
y_test_pred = model.predict(X_test)
print("Métricas no Teste:")
print(classification_report(y_test, y_test_pred, target_names=y_le.classes_))
print(confusion_matrix(y_test, y_test_pred))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   history_id              3000 non-null   object
 1   from_stage              3000 non-null   object
 2   to_stage                3000 non-null   object
 3   moved_at                3000 non-null   object
 4   total_moves             3000 non-null   int64 
 5   days_since_creation     3000 non-null   int64 
 6   current_stage_duration  3000 non-null   int64 
 7   value                   3000 non-null   int64 
 8   sector                  3000 non-null   object
 9   prioridade              3000 non-null   object
 10  target                  3000 non-null   object
dtypes: int64(4), object(7)
memory usage: 257.9+ KB
None
target
pouco provável    1000
muito provável    1000
provável          1000
Name: count, dtype: int64
Métricas no Treino:
                precision    recall  f

In [20]:
# 7. Exemplos de uso

exemplo_muito_provavel = {
    'from_stage': 'Conversa com o Cliente',
    'to_stage': 'Fechamento',
    'total_moves': 7,
    'days_since_creation': 45,
    'current_stage_duration': 10,
    'value': 50000,
    'sector': 'Tecnologia',
    'prioridade': 'Alta'
}

exemplo_pouco_provavel = {
    'from_stage': 'Análise de Perfil',
    'to_stage': 'Conversa com o Cliente',
    'total_moves': 1,
    'days_since_creation': 2,
    'current_stage_duration': 2,
    'value': 500,
    'sector': 'Educação',
    'prioridade': 'Baixa'
}

exemplo_provavel = {
    'from_stage': 'Conversa com o Cliente',
    'to_stage': 'Negociação',
    'total_moves': 3,
    'days_since_creation': 15,
    'current_stage_duration': 5,
    'value': 12000,
    'sector': 'Serviços',
    'prioridade': 'Média'
}

exemplos = [exemplo_muito_provavel, exemplo_pouco_provavel, exemplo_provavel]
df_exemplos = pd.DataFrame(exemplos)

X_cat_ex = ohe.transform(df_exemplos[cat_cols])
X_num_ex = scaler.transform(df_exemplos[num_cols])
X_ex_final = np.hstack([X_cat_ex, X_num_ex])

y_pred_ex = model.predict(X_ex_final)
y_pred_ex_label = y_le.inverse_transform(y_pred_ex)

for i, exemplo in enumerate(exemplos):
    print(f"\nExemplo {i+1}:")
    print(exemplo)
    print("Predição do modelo:", y_pred_ex_label[i])


Exemplo 1:
{'from_stage': 'Conversa com o Cliente', 'to_stage': 'Fechamento', 'total_moves': 7, 'days_since_creation': 45, 'current_stage_duration': 10, 'value': 50000, 'sector': 'Tecnologia', 'prioridade': 'Alta'}
Predição do modelo: muito provável

Exemplo 2:
{'from_stage': 'Análise de Perfil', 'to_stage': 'Conversa com o Cliente', 'total_moves': 1, 'days_since_creation': 2, 'current_stage_duration': 2, 'value': 500, 'sector': 'Educação', 'prioridade': 'Baixa'}
Predição do modelo: pouco provável

Exemplo 3:
{'from_stage': 'Conversa com o Cliente', 'to_stage': 'Negociação', 'total_moves': 3, 'days_since_creation': 15, 'current_stage_duration': 5, 'value': 12000, 'sector': 'Serviços', 'prioridade': 'Média'}
Predição do modelo: provável


In [ ]:
# 3. Salvar como .pkl
with open("rf_model.pkl", "wb") as f:
    pickle.dump(model, f)

# 4. Carregar o modelo do .pkl
with open("rf_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

# 5. Converter para ONNX
initial_type = [("input", FloatTensorType([None, X.shape[1]]))]
onnx_model = convert_sklearn(loaded_model, initial_types=initial_type)

# 6. Salvar modelo ONNX
with open("rf_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())